# Chapter 2 Lab: Metrics, Thresholds, and Tradeoffs

This lab explores the rich landscape of metrics and thresholds. We'll sweep thresholds to understand precision-recall tradeoffs, visualize ROC curves, check calibration, and optimize for business costs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.calibration import calibration_curve

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## Section 1: Threshold Sweep

Logistic regression gives us predicted probabilities. The threshold at which we switch from 0→1 controls the precision-recall tradeoff.

Let's sweep thresholds from 0.1 to 0.9 and see how precision, recall, and F1 change.

In [ ]:
# Generate a synthetic two-class problem
X, y = make_moons(n_samples=300, noise=0.15, random_state=42)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train logistic regression
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Get predicted probabilities (not binary predictions)
y_proba = model.predict_proba(X_test)[:, 1]  # P(y=1)

print(f"Test set size: {len(X_test)}")
print(f"True class distribution: {np.sum(y_test == 1)} positive, {np.sum(y_test == 0)} negative")
print(f"\nPredicted probabilities range: [{y_proba.min():.3f}, {y_proba.max():.3f}]")

In [ ]:
# Sweep thresholds
thresholds = np.linspace(0.1, 0.9, 20)
precisions = []
recalls = []
f1_scores = []

for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    
    # Handle edge cases where a class might not appear
    if np.sum(y_pred) == 0:
        prec = 0.0
    else:
        prec = precision_score(y_test, y_pred, zero_division=0)
    
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)

print("Threshold Sweep Results:")
print("-" * 70)
print(f"{'Threshold':<12} {'Precision':<15} {'Recall':<15} {'F1 Score':<15}")
print("-" * 70)
for i, thresh in enumerate(thresholds[::3]):  # Print every 3rd row to save space
    idx = i * 3
    print(f"{thresh:<12.2f} {precisions[idx]:<15.3f} {recalls[idx]:<15.3f} {f1_scores[idx]:<15.3f}")
print("-" * 70)

## Section 2: Visualizing Precision-Recall-F1 Tradeoffs

In [ ]:
# Plot all three metrics vs threshold
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(thresholds, precisions, marker='o', linewidth=2.5, label='Precision', color='#E63946', markersize=6)
ax.plot(thresholds, recalls, marker='s', linewidth=2.5, label='Recall', color='#2A9D8F', markersize=6)
ax.plot(thresholds, f1_scores, marker='^', linewidth=2.5, label='F1 Score', color='#F77F00', markersize=6)

# Mark the threshold of max F1
best_f1_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_f1_idx]
ax.axvline(best_thresh, color='gray', linestyle='--', alpha=0.5, linewidth=1.5, label=f'Max F1 at threshold={best_thresh:.2f}')

ax.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
ax.set_ylabel('Metric Score', fontsize=12, fontweight='bold')
ax.set_title('Threshold Tradeoffs: Precision, Recall, F1', fontsize=13, fontweight='bold')
ax.set_xlim([0.05, 0.95])
ax.set_ylim([0, 1.05])
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\nOptimal threshold (max F1): {best_thresh:.3f}")
print(f"  Precision: {precisions[best_f1_idx]:.3f}")
print(f"  Recall: {recalls[best_f1_idx]:.3f}")
print(f"  F1: {f1_scores[best_f1_idx]:.3f}")

## Section 3: ROC Curve and AUC

The ROC curve plots True Positive Rate (sensitivity) vs False Positive Rate (1 - specificity). AUC summarizes it into a single number: probability that a random positive is ranked higher than a random negative.

In [ ]:
# Compute ROC curve and AUC
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

# Also compute for a random classifier (baseline)
random_proba = np.random.rand(len(y_test))
fpr_random, tpr_random, _ = roc_curve(y_test, random_proba)
roc_auc_random = auc(fpr_random, tpr_random)

print(f"Logistic Regression AUC: {roc_auc:.3f}")
print(f"Random Classifier AUC: {roc_auc_random:.3f}")
print(f"\nAUC interpretation: Our model is better than random by {(roc_auc - 0.5)*100:.1f} percentage points.")

In [ ]:
# Plot ROC curve
fig, ax = plt.subplots(1, 1, figsize=(9, 8))

# Plot our model's ROC curve
ax.plot(fpr, tpr, color='#2E86AB', lw=2.5, label=f'Logistic Regression (AUC = {roc_auc:.3f})')

# Plot random classifier baseline
ax.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier (AUC = 0.500)')

# Plot diagonal perfect classifier (upper left corner)
ax.fill_between(fpr, tpr, alpha=0.1, color='#2E86AB')

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve: Trade-off Between Sensitivity and Specificity', fontsize=13, fontweight='bold')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.legend(fontsize=11, loc='lower right')
ax.grid(alpha=0.3, linestyle='--')

# Annotate a few points on the curve
for i in range(0, len(fpr), len(fpr)//5):
    if i < len(roc_thresholds):
        ax.plot(fpr[i], tpr[i], 'ro', markersize=5, alpha=0.5)

plt.tight_layout()
plt.show()

## Section 4: Calibration

A well-calibrated classifier outputs probabilities that match empirical frequencies. For example, if we predict P(y=1)=0.7 on 100 examples, about 70 of them should have y=1.

In [ ]:
# Compute calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)

# Also create a poorly calibrated model by scaling probabilities
y_proba_scaled = np.power(y_proba, 0.5)  # Exaggerate probabilities
prob_true_scaled, prob_pred_scaled = calibration_curve(y_test, y_proba_scaled, n_bins=10)

print("Logistic Regression Calibration:")
print(f"  Perfect calibration: prob_true ≈ prob_pred")
print(f"  Our model: average deviation = {np.mean(np.abs(prob_true - prob_pred)):.3f}")
print(f"\nPoorly Calibrated Model (squared probabilities):")
print(f"  Average deviation = {np.mean(np.abs(prob_true_scaled - prob_pred_scaled)):.3f}")

In [ ]:
# Plot calibration curves
fig, ax = plt.subplots(1, 1, figsize=(9, 8))

# Perfect calibration line
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=2, label='Perfect Calibration')

# Well-calibrated model
ax.plot(prob_pred, prob_true, marker='o', linewidth=2.5, markersize=8,
        color='#2A9D8F', label='Logistic Regression (well-calibrated)')

# Poorly calibrated model
ax.plot(prob_pred_scaled, prob_true_scaled, marker='s', linewidth=2.5, markersize=8,
        color='#E63946', label='Poorly Calibrated Model')

ax.set_xlabel('Mean Predicted Probability', fontsize=12, fontweight='bold')
ax.set_ylabel('Fraction of Positives', fontsize=12, fontweight='bold')
ax.set_title('Calibration Curves: Do Predicted Probabilities Match Reality?', fontsize=13, fontweight='bold')
ax.set_xlim([-0.05, 1.05])
ax.set_ylim([-0.05, 1.05])
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Points close to the diagonal mean the model is well-calibrated.")
print("- Points above the diagonal mean the model is overconfident (says 0.7 but happens 0.9 of the time).")
print("- Points below the diagonal mean the model is underconfident.")

## Section 5: Cost-Sensitive Thresholds

In real business problems, false positives and false negatives have different costs. Let's find the optimal threshold that minimizes expected cost.

In [ ]:
# Define costs
cost_fp = 1.0   # False positive: $1 (e.g., customer contacts us to complain)
cost_fn = 10.0  # False negative: $10 (e.g., real spam gets through, customer unhappy)

print(f"Business Model:")
print(f"  Cost of False Positive (FP): ${cost_fp:.2f}")
print(f"  Cost of False Negative (FN): ${cost_fn:.2f}")
print(f"  Ratio: FN/FP = {cost_fn/cost_fp:.1f}x\n")
print(f"Interpretation: Missing spam is {cost_fn/cost_fp:.0f}x worse than a false alarm.\n")

# Sweep thresholds and compute expected cost
thresholds_fine = np.linspace(0.01, 0.99, 100)
costs = []

for thresh in thresholds_fine:
    y_pred = (y_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    
    # Expected cost
    total_cost = (fp * cost_fp) + (fn * cost_fn)
    costs.append(total_cost)

costs = np.array(costs)
best_cost_idx = np.argmin(costs)
best_thresh_cost = thresholds_fine[best_cost_idx]

print(f"Optimal threshold (min cost): {best_thresh_cost:.3f}")
print(f"Expected cost at optimal threshold: ${costs[best_cost_idx]:.2f}")

# Compare to default threshold (0.5)
y_pred_default = (y_proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_default, labels=[0, 1]).ravel()
cost_default = (fp * cost_fp) + (fn * cost_fn)
print(f"\nCost at default threshold (0.5): ${cost_default:.2f}")
print(f"Savings from optimal threshold: ${cost_default - costs[best_cost_idx]:.2f}")

In [ ]:
# Plot cost vs threshold
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot(thresholds_fine, costs, linewidth=2.5, color='#2E86AB', label='Expected Cost')
ax.axvline(best_thresh_cost, color='#2A9D8F', linestyle='--', linewidth=2, label=f'Optimal Threshold = {best_thresh_cost:.3f}')
ax.axvline(0.5, color='#E63946', linestyle=':', linewidth=2, alpha=0.7, label='Default Threshold = 0.50')

ax.scatter([best_thresh_cost], [costs[best_cost_idx]], s=150, color='#2A9D8F', marker='*', zorder=5, edgecolor='black', linewidth=1.5)
ax.scatter([0.5], [cost_default], s=100, color='#E63946', marker='o', zorder=5, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Decision Threshold', fontsize=12, fontweight='bold')
ax.set_ylabel('Expected Cost ($)', fontsize=12, fontweight='bold')
ax.set_title(f'Cost-Sensitive Threshold Selection (FN cost = {cost_fn:.0f}x FP cost)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## What to Notice

1. **Thresholds control tradeoffs**: Lowering the threshold increases recall but decreases precision. There's no free lunch.

2. **ROC-AUC is threshold-independent**: It summarizes performance across all possible thresholds. A higher AUC means the model ranks positives higher than negatives, on average.

3. **Calibration matters**: A model can have high AUC but terrible calibration. Always check if predicted probabilities match empirical frequencies.

4. **Business costs set the threshold**: The "optimal" threshold depends on the costs of FP and FN. If FN is 10x more expensive, we should be more conservative and lower the threshold.

5. **F1 score is a compromise**: When you don't have explicit costs, F1 is a balanced metric between precision and recall. But always check if balance is what you actually want.

6. **Domain knowledge wins**: In practice, talk to domain experts (email users, fraud teams, doctors) to set thresholds. Let the data inform, but let business reality decide.